<a href="https://colab.research.google.com/github/ArushRastogi47/10x.ai/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArushRastogi47/10x.ai/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess
import numpy as np
import pandas as pd
import duckdb

# secret -> env (same as ML-04)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# clone YOUR repo so work/outputs exists
REPO_URL = "https://github.com/ArushRastogi47/10x.ai"
REPO_DIR = "10x.ai"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
os.makedirs("work/outputs", exist_ok=True)

con = duckdb.connect()
con.execute(f"CREATE SECRET IF NOT EXISTS hf_sec (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. My rule and its reason codes

The rule in plain words: a page goes to the top of the refresh queue if it (a) still gets meaningful search visibility in the last 30 days, and (b) that visibility is clearly falling month-over-month. Score = trailing-30d impressions × (1 − momentum), so a big-and-falling page outranks a small-and-falling one, and a page that grew or held steady scores zero. Reason codes say which case a row is; the action label says what a human should do with it.
Two signals checked before trusting the rule:
Signal A (flag-linked — behind FlyRank's CTR-fix logic): CTR should collapse as average position worsens. If it doesn't, position-based reasoning in my queue is on shaky ground.
Signal B (behind refresh prioritization): falling pages should keep falling — the April-decline rate should drop as the Feb→Mar momentum ratio rises. If it doesn't, my "falling" signal isn't predictive and the rule needs rework. A clearly-explained FALSE here saves the rule.
Signal A verdict: TBD from the table below. Signal B verdict: TBD.
Code cell (Signal A — bucket table with n):


In [4]:
FEATURE_SQL = f"""
WITH base AS (
  SELECT client_hash_id, content_hash_id, report_date,
         gsc_impressions, gsc_clicks, gsc_avg_position
  FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-0[2-4]/*.parquet')
  WHERE gsc_data_available IS TRUE
),
feat AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) FILTER (report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31') AS impr_trailing30,
    SUM(gsc_impressions) FILTER (report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28') AS impr_prior30,
    SUM(gsc_clicks)      FILTER (report_date BETWEEN DATE '2026-02-01' AND DATE '2026-03-31') AS clicks_60d,
    SUM(gsc_impressions) FILTER (report_date BETWEEN DATE '2026-02-01' AND DATE '2026-03-31') AS impr_60d,
    AVG(gsc_avg_position) FILTER (report_date BETWEEN DATE '2026-02-01' AND DATE '2026-03-31') AS position_mean_60d,
    COUNT(DISTINCT report_date) AS days_seen_60d
  FROM base
  WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-03-31'
  GROUP BY 1, 2
),
outcome AS (
  SELECT content_hash_id,
         SUM(gsc_impressions) AS impr_apr
  FROM base
  WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
  GROUP BY 1
)
SELECT f.*, COALESCE(o.impr_apr, 0) AS impr_apr,
       (COALESCE(o.impr_apr, 0) < f.impr_trailing30)::INT AS label_declined_apr
FROM feat f LEFT JOIN outcome o USING (content_hash_id)
"""

ff = con.sql(FEATURE_SQL).df()
ff["ctr_60d"] = ff["clicks_60d"] / ff["impr_60d"].replace(0, np.nan)
ff["momentum_30d"] = ff["impr_trailing30"] / ff["impr_prior30"].replace(0, np.nan)
ff.to_csv("work/outputs/w03_feature_frame_2026_03.csv", index=False)  # local cache, not committed
print(len(ff), "pages;", ff["label_declined_apr"].mean().round(3), "April-decline rate")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

196059 pages; 0.634 April-decline rate


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ff = pd.read_csv("work/outputs/w03_feature_frame_2026_03.csv")  # or rebuild via FEATURE_SQL from ML-04 if not cached
a = ff.dropna(subset=["position_mean_60d", "ctr_60d"]).copy()
a["pos_tier"] = pd.cut(a["position_mean_60d"], [0, 3, 5, 10, 20, 100],
                       labels=["1-3", "3-5", "5-10", "10-20", "20+"])
(a.groupby("pos_tier", observed=True)
  .agg(n=("content_hash_id", "size"),
       mean_ctr_pct=("ctr_60d", lambda s: 100 * s.mean()),
       median_ctr_pct=("ctr_60d", lambda s: 100 * s.median()))
  .reset_index())

,pos_tier,n,mean_ctr_pct,median_ctr_pct
0,1-3,16689,1.072881,0.064316
1,3-5,28703,0.723415,0.096455
2,5-10,64625,0.467050,0.000000
3,10-20,39460,0.340044,0.000000
4,20+,44902,0.198374,0.000000


In [6]:
b = ff.dropna(subset=["momentum_30d", "label_declined_apr"]).copy()
b["mom_band"] = pd.cut(b["momentum_30d"], [0, .25, .5, .75, 1.0, 2.0, np.inf],
                       labels=["<0.25", "0.25-0.5", "0.5-0.75", "0.75-1.0", "1.0-2.0", ">2.0"])
(b.groupby("mom_band", observed=True)
  .agg(n=("content_hash_id", "size"),
       apr_decline_rate=("label_declined_apr", "mean"))
  .reset_index())

,mom_band,n,apr_decline_rate
0,<0.25,4853,0.599217
1,0.25-0.5,8398,0.640510
2,0.5-0.75,11055,0.673541
3,0.75-1.0,18304,0.690450
4,1.0-2.0,48585,0.720078
5,>2.0,43043,0.653742


## 2. Build the ranked queue (writes the CSV)
Rule inputs are trailing-window measurements only (Feb–Mar). The April outcome (label_declined_apr) is used only after the queue is built, to compute precision@K — the same evaluation contract as notebook 02. It is never a rule input.

In [8]:
VISIBILITY_FLOOR = 100   # justify from the printed quantiles below
q = ff["impr_trailing30"].quantile([0.5, 0.75, 0.9]).round(0)
print(q)  # choose the floor from this, then freeze it

queue = ff.dropna(subset=["momentum_30d"]).copy()
queue["score"] = queue["impr_trailing30"] * (1 - queue["momentum_30d"]).clip(lower=0)

conditions = [
    (queue["impr_trailing30"] >= VISIBILITY_FLOOR) & (queue["momentum_30d"] < 1.0),
    (queue["impr_trailing30"] <  VISIBILITY_FLOOR) & (queue["momentum_30d"] < 1.0),
]
queue["reason_code"] = np.select(conditions,
                                 ["FALLING_HIGH_VOLUME", "FALLING_LOW_VOLUME"],
                                 default="NOT_DECLINING")
queue["action"] = np.select(
    [queue["reason_code"].eq("FALLING_HIGH_VOLUME"),
     queue["reason_code"].eq("FALLING_LOW_VOLUME")],
    ["REFRESH_NOW", "REVIEW"], default="MONITOR")
queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

queue[["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action",
       "impr_trailing30", "impr_prior30", "momentum_30d", "position_mean_60d",
       "ctr_60d", "days_seen_60d"]].to_csv("work/outputs/baseline_action_score.csv", index=False)

# evaluate on the same label the Week-5 model will use (post-hoc only)
# from sklearn.metrics import ...  # This line caused the error; commented out
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

labeled = queue.dropna(subset=["label_declined_apr"])
metrics = {
    "n_scored": int(len(queue)),
    "base_rate_april_decline": round(float(labeled["label_declined_apr"].mean()), 4),
    "precision_at_20": precision_at_k(labeled["score"], labeled["label_declined_apr"], 20),
    "precision_at_50": precision_at_k(labeled["score"], labeled["label_declined_apr"], 50),
    "visibility_floor": VISIBILITY_FLOOR,
}
print(metrics)
import json
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)


0.50     173.0
0.75    1039.0
0.90    3930.0
Name: impr_trailing30, dtype: float64
{'n_scored': 134238, 'base_rate_april_decline': 0.6816, 'precision_at_20': 0.65, 'precision_at_50': 0.64, 'visibility_floor': 100}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["rank", "score", "reason_code", "action", "impr_trailing30",
        "impr_prior30", "momentum_30d", "position_mean_60d", "ctr_60d", "days_seen_60d"]
queue.head(20)[cols]

,rank,score,reason_code,action,impr_trailing30,impr_prior30,momentum_30d,position_mean_60d,ctr_60d,days_seen_60d
0,1,47911.631481,FALLING_HIGH_VOLUME,REFRESH_NOW,83834.0,195648.0,0.428494,7.063271,0.000007,59
1,2,45403.908181,FALLING_HIGH_VOLUME,REFRESH_NOW,134984.0,203401.0,0.663635,4.745605,0.000009,59
2,3,44702.542484,FALLING_HIGH_VOLUME,REFRESH_NOW,124075.0,193954.0,0.639714,6.755773,0.000003,59
3,4,35346.690954,FALLING_HIGH_VOLUME,REFRESH_NOW,65681.0,142215.0,0.461843,2.169411,0.011275,57
4,5,32297.472020,FALLING_HIGH_VOLUME,REFRESH_NOW,60919.0,129662.0,0.469829,1.809635,0.011858,59
5,6,27763.261498,FALLING_HIGH_VOLUME,REFRESH_NOW,65736.0,113798.0,0.577655,3.471272,0.005453,59
6,7,26101.159628,FALLING_HIGH_VOLUME,REFRESH_NOW,91408.0,127941.0,0.714454,3.761689,0.003041,59
7,8,20168.038438,FALLING_HIGH_VOLUME,REFRESH_NOW,45617.0,81768.0,0.557883,6.828671,0.000518,59
8,9,18940.115210,FALLING_HIGH_VOLUME,REFRESH_NOW,142304.0,164152.0,0.866904,3.109872,0.002428,59
9,10,18846.277140,FALLING_HIGH_VOLUME,REFRESH_NOW,72918.0,98333.0,0.741541,3.196135,0.005530,59


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Your honest read on which top rows look wrong and why (tiny denominators, sparse history, position outliers). Then the leakage confirmation below.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# weak picks inside the top 50: unstable momentum or sparse history
top50 = queue.head(50)
suspect = top50[(top50["impr_prior30"] < 30) | (top50["days_seen_60d"] < 20)]
suspect[["rank", "score", "impr_trailing30", "impr_prior30", "momentum_30d", "days_seen_60d"]]

,rank,score,impr_trailing30,impr_prior30,momentum_30d,days_seen_60d


In [11]:
# leakage check: the rule must run on a frame with NO outcome-window columns
RULE_INPUTS = ["impr_trailing30", "impr_prior30",  # -> momentum_30d is derived from these
               "momentum_30d"]
OUTCOME_COLS  = ["impr_apr", "label_declined_apr"]
assert set(RULE_INPUTS).isdisjoint(OUTCOME_COLS)

label_free = ff.drop(columns=OUTCOME_COLS)
# re-derive momentum and rebuild the queue from the label-free frame:
lf = label_free.dropna(subset=["momentum_30d"]).copy()
lf["score"] = lf["impr_trailing30"] * (1 - lf["momentum_30d"]).clip(lower=0)
pd.testing.assert_series_equal(
    lf.sort_values("score", ascending=False)["content_hash_id"].head(100).reset_index(drop=True),
    queue.head(100)["content_hash_id"].reset_index(drop=True),
    check_names=False)
print("leakage check passed: top-100 queue identical with all outcome columns dropped")

leakage check passed: top-100 queue identical with all outcome columns dropped


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.